# MMIS Market Basket Analysis

This notebook implements Market Basket Analysis using the Multiple Minimum Item Support (MMIS) algorithm on online retail transaction data. The analysis segments transactions by time of day and discovers association rules using dynamically calculated minimum support thresholds.

## Table of Contents
1. Data Loading and Preprocessing
2. Time-Based Transaction Segmentation
3. One-Hot Encoding
4. MIS Value Calculation
5. MMIS Algorithm Implementation
6. Experimental Configurations (Exp 1 & Exp 2)
7. Association Rule Generation
8. Rule Analysis and Reporting
9. Comparative Analysis

## 1. Data Loading and Preprocessing

First, we'll import necessary libraries and load the online retail dataset.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
from itertools import combinations, chain
from collections import defaultdict

# Load the dataset
df = pd.read_csv('Rida Zubair - online_retail - online_retail.csv', encoding='ISO-8859-1')
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

### Data Cleaning

Remove cancelled invoices (starting with 'C') and records with missing descriptions.

In [ ]:
# Clean the data
print(f"Original dataset size: {len(df)}")

# Create synthetic InvoiceNo by combining CustomerID and InvoiceDate
# Since the dataset doesn't have InvoiceNo, we group by customer and timestamp
df['InvoiceNo'] = df['CustomerID'].astype(str) + '_' + df['InvoiceDate'].astype(str)
print(f"Created {df['InvoiceNo'].nunique()} unique invoice numbers")

# Remove cancelled invoices (if any InvoiceNo starts with 'C')
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
print(f"After removing cancelled invoices: {len(df)}")

# Remove records with missing descriptions
df = df[df['Description'].notna()]
print(f"After removing missing descriptions: {len(df)}")

# Convert InvoiceDate to datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(f"\nDate range: {df['InvoiceDate'].min()} to {df['InvoiceDate'].max()}")

### Group Items into Transaction Baskets

Group items by InvoiceNo to create transaction lists.

In [ ]:
# Group items by InvoiceNo to create baskets
transactions = df.groupby('InvoiceNo')['Description'].apply(list).reset_index()
transactions.columns = ['InvoiceNo', 'Items']

# Merge with InvoiceDate (take first occurrence per invoice)
invoice_dates = df.groupby('InvoiceNo')['InvoiceDate'].first().reset_index()
transactions = transactions.merge(invoice_dates, on='InvoiceNo')

print(f"Total transactions: {len(transactions)}")
print(f"\nSample transactions:")
for i in range(3):
    print(f"\nTransaction {i+1}: {transactions.iloc[i]['Items'][:5]}...")

## 2. Time-Based Transaction Segmentation

Segment transactions into four time groups based on the hour of purchase:
- Morning: 5-11
- Afternoon: 12-16
- Evening: 17-20
- Night: 21-23, 0-4

In [ ]:
def assign_time_group(hour):
    """Assign time group based on hour of day."""
    if 5 <= hour <= 11:
        return 'Morning'
    elif 12 <= hour <= 16:
        return 'Afternoon'
    elif 17 <= hour <= 20:
        return 'Evening'
    else:  # 21-23 or 0-4
        return 'Night'

# Extract hour and assign time group
transactions['Hour'] = transactions['InvoiceDate'].dt.hour
transactions['TimeGroup'] = transactions['Hour'].apply(assign_time_group)

# Display time group statistics
print("Transaction counts by time group:")
print(transactions['TimeGroup'].value_counts().sort_index())

# Create separate transaction lists for each time group
time_groups = {}
for group in ['Morning', 'Afternoon', 'Evening', 'Night']:
    time_groups[group] = transactions[transactions['TimeGroup'] == group]['Items'].tolist()
    print(f"\n{group}: {len(time_groups[group])} transactions")
    if len(time_groups[group]) > 0:
        print(f"  Sample: {time_groups[group][0][:3]}...")

## 3. One-Hot Encoding

Convert transaction baskets into one-hot encoded format for frequent itemset mining.

In [ ]:
from mlxtend.preprocessing import TransactionEncoder

def encode_transactions(transaction_list):
    """Convert transaction list to one-hot encoded DataFrame."""
    te = TransactionEncoder()
    te_array = te.fit(transaction_list).transform(transaction_list)
    df_encoded = pd.DataFrame(te_array, columns=te.columns_)
    return df_encoded

# Encode transactions for each time group
encoded_groups = {}
for group_name, trans_list in time_groups.items():
    if len(trans_list) > 0:
        encoded_groups[group_name] = encode_transactions(trans_list)
        print(f"\n{group_name}:")
        print(f"  Shape: {encoded_groups[group_name].shape}")
        print(f"  Unique items: {encoded_groups[group_name].shape[1]}")
        print(f"  Sample (first 3 items):")
        print(encoded_groups[group_name].iloc[:3, :3])

## 4. MIS Value Calculation

Calculate dynamic minimum support values for each item using the formula:
MIS(i) = max(β × support(i), LS)

This allows rare items to have lower support thresholds than common items.

In [ ]:
def calculate_mis(df_encoded, beta, ls):
    """
    Calculate MIS values for all items.
    
    Parameters:
    - df_encoded: One-hot encoded DataFrame
    - beta: Beta parameter (0 < β ≤ 1)
    - ls: Least Support threshold
    
    Returns:
    - Dictionary mapping item -> MIS value
    """
    n_transactions = len(df_encoded)
    mis_values = {}
    
    for item in df_encoded.columns:
        support = df_encoded[item].sum() / n_transactions
        mis = max(beta * support, ls)
        mis_values[item] = mis
    
    return mis_values

# Test MIS calculation with sample parameters
sample_group = 'Morning'
if sample_group in encoded_groups:
    test_mis = calculate_mis(encoded_groups[sample_group], beta=0.5, ls=0.01)
    
    # Calculate support for display
    n_trans = len(encoded_groups[sample_group])
    item_support = {item: encoded_groups[sample_group][item].sum() / n_trans 
                    for item in encoded_groups[sample_group].columns}
    
    # Display top 10 items by support
    sorted_items = sorted(item_support.items(), key=lambda x: x[1], reverse=True)[:10]
    
    print(f"Top 10 items in {sample_group} (β=0.5, LS=0.01):")
    print(f"{'Item':<40} {'Support':<10} {'MIS'}")
    print("-" * 65)
    for item, supp in sorted_items:
        print(f"{item[:40]:<40} {supp:<10.4f} {test_mis[item]:.4f}")

## 5. MMIS Algorithm Implementation

Implement the MMIS (Multiple Minimum Item Support) algorithm with level-wise candidate generation.

Key differences from standard Apriori:
- Each item has its own minimum support threshold (MIS value)
- Items are sorted by MIS values
- Candidate itemsets are checked against the minimum MIS of their items

In [ ]:
def calculate_support(df_encoded, itemset):
    """Calculate support for an itemset."""
    if len(itemset) == 1:
        return df_encoded[list(itemset)[0]].sum() / len(df_encoded)
    else:
        mask = df_encoded[list(itemset)].all(axis=1)
        return mask.sum() / len(df_encoded)

def mmis_algorithm(df_encoded, beta, ls):
    """
    MMIS frequent itemset mining algorithm.
    
    Parameters:
    - df_encoded: One-hot encoded DataFrame
    - beta: Beta parameter for MIS calculation
    - ls: Least Support threshold
    
    Returns:
    - Dictionary of frequent itemsets with their support values
    """
    # Step 1: Calculate MIS values for all items
    mis_values = calculate_mis(df_encoded, beta, ls)
    
    # Step 2: Sort items by MIS values (ascending)
    sorted_items = sorted(mis_values.items(), key=lambda x: x[1])
    item_order = [item for item, _ in sorted_items]
    
    # Step 3: Generate frequent 1-itemsets
    frequent_itemsets = {}
    L1 = []
    
    for item in item_order:
        support = calculate_support(df_encoded, {item})
        if support >= mis_values[item]:
            itemset = frozenset([item])
            frequent_itemsets[itemset] = support
            L1.append(itemset)
    
    print(f"  Found {len(L1)} frequent 1-itemsets")
    
    # Step 4: Level-wise generation of k-itemsets (k > 1)
    k = 2
    Lk_prev = L1
    
    while len(Lk_prev) > 0:
        # Generate candidates
        candidates = set()
        
        # Join step: combine itemsets that share first k-2 items
        for i, itemset1 in enumerate(Lk_prev):
            for itemset2 in Lk_prev[i+1:]:
                # Convert to sorted lists for comparison
                items1 = sorted(list(itemset1), key=lambda x: item_order.index(x))
                items2 = sorted(list(itemset2), key=lambda x: item_order.index(x))
                
                # Check if first k-2 items are the same
                if k == 2 or items1[:-1] == items2[:-1]:
                    # Create new candidate
                    new_candidate = frozenset(items1 + [items2[-1]])
                    if len(new_candidate) == k:
                        candidates.add(new_candidate)
        
        # Prune step: remove candidates with infrequent subsets
        valid_candidates = []
        for candidate in candidates:
            # Check all (k-1)-subsets
            subsets = [frozenset(s) for s in combinations(candidate, k-1)]
            if all(subset in frequent_itemsets for subset in subsets):
                valid_candidates.append(candidate)
        
        # Check support for valid candidates
        Lk = []
        for candidate in valid_candidates:
            support = calculate_support(df_encoded, candidate)
            # Use minimum MIS of items in the candidate
            min_mis = min(mis_values[item] for item in candidate)
            if support >= min_mis:
                frequent_itemsets[candidate] = support
                Lk.append(candidate)
        
        if len(Lk) > 0:
            print(f"  Found {len(Lk)} frequent {k}-itemsets")
        
        Lk_prev = Lk
        k += 1
    
    return frequent_itemsets

## 6. Experimental Configurations

Run two experiments with different parameter settings:
- Experiment 1: β=0.5, LS=0.01
- Experiment 2: β=0.7, LS=0.02

### Experiment 1: β=0.5, LS=0.01

In [ ]:
# Run Experiment 1
exp1_results = {}
beta1, ls1 = 0.5, 0.01

print("Running Experiment 1 (β=0.5, LS=0.01)...\n")

for group_name, df_encoded in encoded_groups.items():
    print(f"\nProcessing {group_name}...")
    frequent_itemsets = mmis_algorithm(df_encoded, beta1, ls1)
    exp1_results[group_name] = frequent_itemsets
    
    # Summary statistics
    total_itemsets = len(frequent_itemsets)
    max_length = max(len(itemset) for itemset in frequent_itemsets.keys()) if frequent_itemsets else 0
    
    print(f"\n{group_name} Summary:")
    print(f"  Total frequent itemsets: {total_itemsets}")
    print(f"  Largest itemset size: {max_length}")
    
    # Top 5 by support
    sorted_itemsets = sorted(frequent_itemsets.items(), key=lambda x: x[1], reverse=True)[:5]
    print(f"\n  Top 5 frequent itemsets by support:")
    for i, (itemset, support) in enumerate(sorted_itemsets, 1):
        items_str = ', '.join(list(itemset)[:3])  # Show first 3 items
        if len(itemset) > 3:
            items_str += f"... ({len(itemset)} items)"
        print(f"    {i}. {{{items_str}}} - Support: {support:.4f}")

### Experiment 2: β=0.7, LS=0.02

In [ ]:
# Run Experiment 2
exp2_results = {}
beta2, ls2 = 0.7, 0.02

print("Running Experiment 2 (β=0.7, LS=0.02)...\n")

for group_name, df_encoded in encoded_groups.items():
    print(f"\nProcessing {group_name}...")
    frequent_itemsets = mmis_algorithm(df_encoded, beta2, ls2)
    exp2_results[group_name] = frequent_itemsets
    
    # Summary statistics
    total_itemsets = len(frequent_itemsets)
    max_length = max(len(itemset) for itemset in frequent_itemsets.keys()) if frequent_itemsets else 0
    
    print(f"\n{group_name} Summary:")
    print(f"  Total frequent itemsets: {total_itemsets}")
    print(f"  Largest itemset size: {max_length}")
    
    # Top 5 by support
    sorted_itemsets = sorted(frequent_itemsets.items(), key=lambda x: x[1], reverse=True)[:5]
    print(f"\n  Top 5 frequent itemsets by support:")
    for i, (itemset, support) in enumerate(sorted_itemsets, 1):
        items_str = ', '.join(list(itemset)[:3])  # Show first 3 items
        if len(itemset) > 3:
            items_str += f"... ({len(itemset)} items)"
        print(f"    {i}. {{{items_str}}} - Support: {support:.4f}")

## 7. Association Rule Generation

Generate association rules from frequent itemsets with:
- Confidence ≥ 0.6
- Lift > 1

In [ ]:
def generate_association_rules(frequent_itemsets, min_confidence=0.6, min_lift=1.0):
    """
    Generate association rules from frequent itemsets.
    
    Parameters:
    - frequent_itemsets: Dictionary of itemsets with support values
    - min_confidence: Minimum confidence threshold
    - min_lift: Minimum lift threshold
    
    Returns:
    - List of rules as dictionaries with antecedent, consequent, support, confidence, lift
    """
    rules = []
    
    # Only consider itemsets with 2+ items
    for itemset, support_union in frequent_itemsets.items():
        if len(itemset) < 2:
            continue
        
        # Generate all non-empty antecedent-consequent splits
        for i in range(1, len(itemset)):
            for antecedent in combinations(itemset, i):
                antecedent = frozenset(antecedent)
                consequent = itemset - antecedent
                
                # Calculate confidence and lift
                if antecedent in frequent_itemsets and consequent in frequent_itemsets:
                    support_antecedent = frequent_itemsets[antecedent]
                    support_consequent = frequent_itemsets[consequent]
                    
                    confidence = support_union / support_antecedent
                    lift = support_union / (support_antecedent * support_consequent)
                    
                    # Filter by thresholds
                    if confidence >= min_confidence and lift > min_lift:
                        rules.append({
                            'antecedent': antecedent,
                            'consequent': consequent,
                            'support': support_union,
                            'confidence': confidence,
                            'lift': lift
                        })
    
    return rules

# Generate rules for Experiment 1
exp1_rules = {}
print("Generating association rules for Experiment 1...\n")

for group_name, frequent_itemsets in exp1_results.items():
    rules = generate_association_rules(frequent_itemsets)
    exp1_rules[group_name] = rules
    print(f"{group_name}: {len(rules)} rules generated")

# Generate rules for Experiment 2
exp2_rules = {}
print("\nGenerating association rules for Experiment 2...\n")

for group_name, frequent_itemsets in exp2_results.items():
    rules = generate_association_rules(frequent_itemsets)
    exp2_rules[group_name] = rules
    print(f"{group_name}: {len(rules)} rules generated")

## 8. Rule Analysis and Reporting

Analyze and display the top association rules for each time group.

In [ ]:
def display_top_rules(rules, group_name, top_n=5):
    """
    Display top association rules ranked by lift.
    
    Parameters:
    - rules: List of rule dictionaries
    - group_name: Name of the time group
    - top_n: Number of top rules to display
    """
    print(f"\n{'='*80}")
    print(f"{group_name} - Top {top_n} Association Rules (by Lift)")
    print(f"{'='*80}")
    print(f"Total rules: {len(rules)}\n")
    
    if len(rules) == 0:
        print("No rules found.\n")
        return
    
    # Sort by lift
    sorted_rules = sorted(rules, key=lambda x: x['lift'], reverse=True)[:top_n]
    
    for i, rule in enumerate(sorted_rules, 1):
        ant_str = ', '.join(list(rule['antecedent'])[:2])
        if len(rule['antecedent']) > 2:
            ant_str += f"... ({len(rule['antecedent'])} items)"
        
        cons_str = ', '.join(list(rule['consequent'])[:2])
        if len(rule['consequent']) > 2:
            cons_str += f"... ({len(rule['consequent'])} items)"
        
        print(f"Rule {i}:")
        print(f"  {{{ant_str}}} → {{{cons_str}}}")
        print(f"  Support: {rule['support']:.4f}")
        print(f"  Confidence: {rule['confidence']:.4f}")
        print(f"  Lift: {rule['lift']:.4f}")
        print()

# Display rules for Experiment 1
print("\n" + "#"*80)
print("EXPERIMENT 1 RULES (β=0.5, LS=0.01)")
print("#"*80)

for group_name in ['Morning', 'Afternoon', 'Evening', 'Night']:
    if group_name in exp1_rules:
        display_top_rules(exp1_rules[group_name], group_name)

In [ ]:
# Display rules for Experiment 2
print("\n" + "#"*80)
print("EXPERIMENT 2 RULES (β=0.7, LS=0.02)")
print("#"*80)

for group_name in ['Morning', 'Afternoon', 'Evening', 'Night']:
    if group_name in exp2_rules:
        display_top_rules(exp2_rules[group_name], group_name)

## 9. Comparative Analysis

Perform comparative analysis across experiments and time groups to answer key analytical questions.

### 9.1 Which time group has the strongest association rules?

In [ ]:
# Calculate average lift for each time group (Experiment 1)
print("Average Lift by Time Group (Experiment 1):\n")

avg_lifts_exp1 = {}
for group_name, rules in exp1_rules.items():
    if len(rules) > 0:
        avg_lift = sum(rule['lift'] for rule in rules) / len(rules)
        avg_lifts_exp1[group_name] = avg_lift
        print(f"{group_name}: {avg_lift:.4f} (from {len(rules)} rules)")

if avg_lifts_exp1:
    strongest_group = max(avg_lifts_exp1.items(), key=lambda x: x[1])
    print(f"\n✓ Strongest association rules: {strongest_group[0]} (avg lift: {strongest_group[1]:.4f})")

# Repeat for Experiment 2
print("\n\nAverage Lift by Time Group (Experiment 2):\n")

avg_lifts_exp2 = {}
for group_name, rules in exp2_rules.items():
    if len(rules) > 0:
        avg_lift = sum(rule['lift'] for rule in rules) / len(rules)
        avg_lifts_exp2[group_name] = avg_lift
        print(f"{group_name}: {avg_lift:.4f} (from {len(rules)} rules)")

if avg_lifts_exp2:
    strongest_group = max(avg_lifts_exp2.items(), key=lambda x: x[1])
    print(f"\n✓ Strongest association rules: {strongest_group[0]} (avg lift: {strongest_group[1]:.4f})")

### 9.2 Item combinations appearing across multiple time groups

In [ ]:
# Find frequent itemsets that appear in multiple time groups (Experiment 1)
print("Frequent Itemsets Appearing Across Multiple Time Groups (Experiment 1):\n")

# Collect all itemsets by size
itemset_occurrences = defaultdict(list)

for group_name, frequent_itemsets in exp1_results.items():
    for itemset in frequent_itemsets.keys():
        if len(itemset) >= 2:  # Only consider 2+ item combinations
            itemset_occurrences[itemset].append(group_name)

# Find itemsets appearing in 2+ time groups
cross_group_itemsets = {itemset: groups for itemset, groups in itemset_occurrences.items() 
                        if len(groups) >= 2}

# Sort by number of occurrences
sorted_cross_group = sorted(cross_group_itemsets.items(), key=lambda x: len(x[1]), reverse=True)[:10]

print(f"Found {len(cross_group_itemsets)} itemsets appearing in multiple time groups\n")
print("Top 10 cross-time-group itemsets:\n")

for i, (itemset, groups) in enumerate(sorted_cross_group, 1):
    items_str = ', '.join(list(itemset)[:3])
    if len(itemset) > 3:
        items_str += f"... ({len(itemset)} items)"
    print(f"{i}. {{{items_str}}}")
    print(f"   Appears in: {', '.join(groups)} ({len(groups)} groups)")
    print()

### 9.3 Comparison between Experiment 1 and Experiment 2

In [ ]:
# Compare number of frequent itemsets between experiments
print("Comparison: Experiment 1 (β=0.5, LS=0.01) vs Experiment 2 (β=0.7, LS=0.02)\n")
print(f"{'Time Group':<15} {'Exp 1 Itemsets':<20} {'Exp 2 Itemsets':<20} {'Difference'}")
print("-" * 75)

for group_name in ['Morning', 'Afternoon', 'Evening', 'Night']:
    exp1_count = len(exp1_results.get(group_name, {}))
    exp2_count = len(exp2_results.get(group_name, {}))
    diff = exp1_count - exp2_count
    print(f"{group_name:<15} {exp1_count:<20} {exp2_count:<20} {diff:+d}")

# Calculate totals
total_exp1 = sum(len(itemsets) for itemsets in exp1_results.values())
total_exp2 = sum(len(itemsets) for itemsets in exp2_results.values())
print("-" * 75)
print(f"{'TOTAL':<15} {total_exp1:<20} {total_exp2:<20} {total_exp1 - total_exp2:+d}")

print("\n\nAnalysis:")
print(f"\n1. Experiment 1 discovered {total_exp1} frequent itemsets")
print(f"   Experiment 2 discovered {total_exp2} frequent itemsets")
print(f"   Difference: {total_exp1 - total_exp2:+d} itemsets")

if total_exp1 > total_exp2:
    print("\n2. Experiment 1 found MORE frequent itemsets because:")
    print("   - Lower β (0.5 vs 0.7) means MIS values are lower for all items")
    print("   - Lower LS (0.01 vs 0.02) provides a lower minimum threshold")
    print("   - Both factors make it easier for itemsets to qualify as frequent")
else:
    print("\n2. Experiment 2 found MORE frequent itemsets (unexpected)")
    print("   This could indicate data-specific patterns or edge cases")

### 9.4 How does changing β affect frequent itemset discovery?

In [ ]:
print("Effect of Beta Parameter on Frequent Itemset Discovery:\n")
print("="*80)

print("\nBeta (β) Parameter Role:")
print("-" * 80)
print("The β parameter controls how MIS values scale with item support.")
print("Formula: MIS(i) = max(β × support(i), LS)\n")

print("Lower β (e.g., 0.5):")
print("  • MIS values are lower for all items")
print("  • More items qualify as frequent (lower threshold)")
print("  • More itemsets can be formed from frequent items")
print("  • Result: MORE frequent itemsets discovered\n")

print("Higher β (e.g., 0.7):")
print("  • MIS values are higher for all items")
print("  • Fewer items qualify as frequent (higher threshold)")
print("  • Fewer itemsets can be formed")
print("  • Result: FEWER frequent itemsets discovered\n")

print("Example Calculation:")
print("-" * 80)
print("For an item with support = 0.10:")
print(f"  With β=0.5, LS=0.01: MIS = max(0.5 × 0.10, 0.01) = max(0.05, 0.01) = 0.05")
print(f"  With β=0.7, LS=0.02: MIS = max(0.7 × 0.10, 0.02) = max(0.07, 0.02) = 0.07")
print(f"\n  The item needs 5% support in Exp 1 but 7% support in Exp 2 to be frequent.")
print(f"  This makes it HARDER to qualify in Experiment 2.\n")

print("\nConclusion:")
print("-" * 80)
print("Increasing β makes the algorithm MORE SELECTIVE, discovering fewer but")
print("potentially more significant frequent itemsets. Decreasing β makes it MORE")
print("INCLUSIVE, capturing more patterns including those with lower support.")

### 9.5 MMIS vs Standard Apriori: Key Differences

In [ ]:
print("MMIS vs Standard Apriori Algorithm:\n")
print("="*80)

print("\nStandard Apriori:")
print("-" * 80)
print("• Uses a SINGLE global minimum support threshold (min_sup)")
print("• ALL items must meet the same support threshold")
print("• Problem: Rare but important items may be excluded")
print("• Example: If min_sup = 0.05, an item with 4% support is discarded\n")

print("MMIS (Multiple Minimum Item Support):")
print("-" * 80)
print("• Uses MULTIPLE minimum support thresholds (one per item)")
print("• Each item has its own MIS value: MIS(i) = max(β × support(i), LS)")
print("• Rare items get LOWER thresholds, common items get HIGHER thresholds")
print("• Benefit: Captures rare but meaningful associations\n")

print("Example Scenario:")
print("-" * 80)
print("Consider two items:")
print("  • Item A (common): support = 0.20")
print("  • Item B (rare): support = 0.03\n")

print("Standard Apriori with min_sup = 0.05:")
print("  • Item A: 0.20 ≥ 0.05 ✓ (frequent)")
print("  • Item B: 0.03 < 0.05 ✗ (discarded)")
print("  • Result: Association {A, B} cannot be discovered\n")

print("MMIS with β=0.5, LS=0.01:")
print("  • Item A: MIS = max(0.5 × 0.20, 0.01) = 0.10, support 0.20 ≥ 0.10 ✓")
print("  • Item B: MIS = max(0.5 × 0.03, 0.01) = 0.015, support 0.03 ≥ 0.015 ✓")
print("  • Result: Both items are frequent, {A, B} can be discovered\n")

print("Key Advantage:")
print("-" * 80)
print("MMIS allows discovery of associations involving rare items that would be")
print("missed by standard Apriori. This is crucial in domains like retail where")
print("expensive or specialty items (low support) may have strong associations")
print("with common items (high support).")

## Summary

This notebook successfully implemented the MMIS (Multiple Minimum Item Support) algorithm for Market Basket Analysis on online retail transaction data. Key accomplishments:

1. **Data Processing**: Cleaned and segmented transactions into four time groups (Morning, Afternoon, Evening, Night)

2. **MMIS Implementation**: Built a complete MMIS algorithm from scratch with:
   - Dynamic MIS calculation for each item
   - Level-wise candidate generation
   - Item-specific support thresholds

3. **Experimental Analysis**: Ran two experiments with different parameters:
   - Experiment 1: β=0.5, LS=0.01 (more inclusive)
   - Experiment 2: β=0.7, LS=0.02 (more selective)

4. **Association Rules**: Generated and analyzed rules with confidence ≥ 0.6 and lift > 1

5. **Comparative Insights**:
   - Identified time groups with strongest associations
   - Found cross-time-group patterns
   - Demonstrated how β parameter affects itemset discovery
   - Explained advantages of MMIS over standard Apriori

The MMIS approach successfully captures associations involving both common and rare items, providing richer insights than traditional single-threshold methods.